In [ ]:
# preprocess any file into Indexing set and query set( input will be all the feature vectors,will take 1 from each class for indexing and remaining check for querying)



# Remember name on the plot should be the datset specific
# Implement a function for Just HNSW (Input will be Indexing vectors,Query set) output must be a graph of HR vs PR
# first plot the HR vs PR(varying efsearch) and find the best efsearch, now plot the graph for HR vs Topk(PR) using this efsearch 
# Also calculate Avg time here Total Query time/No of queries

# Implement a function for Just HNSW (Input will be Indexing vectors,Query set) output must be a graph of HR vs PR 
# Also calculate Avg time here (KNN+Total Query time)/No of queries

In [1]:
import os
import time
import re
import math
import heapq
import random
import faiss
from typing import List, Optional
import numpy as np
import plotly.graph_objects as go
from scipy.io import loadmat

In [2]:
class Node:
    def __init__(self, idx: int, vector: np.ndarray, level: int):
        self.idx = idx
        self.vector = vector
        self.level = level
        self.neighbors = [[] for _ in range(level + 1)]
        self.deleted = False


class HNSW:
    def __init__(self, M=16, ef_construction=200, random_seed=None):
        if random_seed is not None:
            random.seed(random_seed)
            np.random.seed(random_seed)
        self.M = M
        self.ef_construction = ef_construction
        self.level_mult = 1.0 / math.log(M) if M > 1 else 1.0
        self.entry_point = None
        self.max_level = -1
        self.nodes = []
        self._visit_mark = 1
        self._visited = {}

    @staticmethod
    def _distance(a, b):
        return float(np.linalg.norm(a - b))

    def _get_random_level(self):
        r = random.random()
        if r == 0.0:
            r = 1e-16
        return int(math.floor(-math.log(r) * self.level_mult))

    def _next_visit(self):
        self._visit_mark += 1
        if self._visit_mark % 1000000 == 0:
            self._visited.clear()

    def _is_visited(self, idx):
        return self._visited.get(idx, 0) == self._visit_mark

    def _set_visited(self, idx):
        self._visited[idx] = self._visit_mark

    def _greedy_search_layer(self, query_vec, entry_idx, level):
        current = entry_idx
        current_dist = self._distance(query_vec, self.nodes[current].vector)
        improved = True
        while improved:
            improved = False
            for nb in self.nodes[current].neighbors[level]:
                d = self._distance(query_vec, self.nodes[nb].vector)
                if d < current_dist:
                    current = nb
                    current_dist = d
                    improved = True
        return current

    def _search_layer(self, query_vec, entry_idx, ef, level):
        candidates = []
        result_heap = []

        self._next_visit()
        entry_dist = self._distance(query_vec, self.nodes[entry_idx].vector)
        heapq.heappush(candidates, (entry_dist, entry_idx))
        heapq.heappush(result_heap, (-entry_dist, entry_idx))
        self._set_visited(entry_idx)

        while candidates:
            top_dist, top_idx = candidates[0]
            worst_dist = -result_heap[0][0]
            if top_dist > worst_dist and len(result_heap) >= ef:
                break
            heapq.heappop(candidates)
            for nb in self.nodes[top_idx].neighbors[level]:
                if self._is_visited(nb):
                    continue
                self._set_visited(nb)
                d = self._distance(query_vec, self.nodes[nb].vector)
                if len(result_heap) < ef or d < -result_heap[0][0]:
                    heapq.heappush(candidates, (d, nb))
                    heapq.heappush(result_heap, (-d, nb))
                    if len(result_heap) > ef:
                        heapq.heappop(result_heap)

        res = [(-dist, idx) for (dist, idx) in result_heap]
        res.sort(key=lambda x: x[0])
        return res

    def _select_neighbors(self, query_vec, candidates, M):
        selected = []
        for dist_q, cand_idx in candidates:
            cand_vec = self.nodes[cand_idx].vector
            ok = True
            for sel_idx in selected:
                if self._distance(cand_vec, self.nodes[sel_idx].vector) < dist_q:
                    ok = False
                    break
            if ok:
                selected.append(cand_idx)
            if len(selected) >= M:
                break
        return selected

    def _select_neighbors_from_ids(self, query_vec, ids, M):
        cand_with_dist = [(self._distance(query_vec, self.nodes[idx].vector), idx) for idx in ids]
        cand_with_dist.sort(key=lambda x: x[0])
        return self._select_neighbors(query_vec, cand_with_dist, M)

    def add_item(self, vector, idx=None):
        if idx is None:
            idx = len(self.nodes)
        vector = np.asarray(vector, dtype=float)
        level = self._get_random_level()
        node = Node(idx, vector, level)
        if idx >= len(self.nodes):
            self.nodes.extend([None] * (idx - len(self.nodes) + 1))
        self.nodes[idx] = node

        if self.entry_point is None:
            self.entry_point = idx
            self.max_level = level
            return

        ep = self.entry_point
        for L in range(self.max_level, level, -1):
            ep = self._greedy_search_layer(vector, ep, L)

        for L in range(min(level, self.max_level), -1, -1):
            candidates = self._search_layer(vector, ep, self.ef_construction, L)
            selected = self._select_neighbors(vector, candidates, self.M)
            node.neighbors[L] = selected.copy()

            for nb in selected:
                nb_neighbors = self.nodes[nb].neighbors[L]
                nb_neighbors.append(idx)
                if len(nb_neighbors) > self.M:
                    pruned = self._select_neighbors_from_ids(self.nodes[nb].vector, nb_neighbors, self.M)
                    self.nodes[nb].neighbors[L] = pruned

            if candidates:
                ep = candidates[0][1]

        if level > self.max_level:
            self.entry_point = idx
            self.max_level = level

    def search_knn(self, query_vec, k=1, ef_search=None):
        if self.entry_point is None:
            return []
        if ef_search is None:
            ef_search = max(k, 50)

        ep = self.entry_point
        for L in range(self.max_level, 0, -1):
            ep = self._greedy_search_layer(query_vec, ep, L)
        results = self._search_layer(query_vec, ep, ef_search, 0)
        return results[:k]


In [3]:
# log2 instead of ln

class Node:
    def __init__(self, idx: int, vector: np.ndarray, level: int):
        self.idx = idx
        self.vector = vector
        self.level = level
        self.neighbors = [[] for _ in range(level + 1)]
        self.deleted = False


class HNSW2:
    def __init__(self, M=16, ef_construction=200, random_seed=None):
        if random_seed is not None:
            random.seed(random_seed)
            np.random.seed(random_seed)
        self.M = M
        self.ef_construction = ef_construction
        
        # UPDATE 1: Scale using log2 instead of math.log (natural log)
        # This aligns the decay rate with our base-2 bitwise operations
        self.bitwise_level_mult = 1.0 / math.log2(M) if M > 1 else 1.0
        
        self.entry_point = None
        self.max_level = -1
        self.nodes = []
        self._visit_mark = 1
        self._visited = {}

    @staticmethod
    def _distance(a, b):
        return float(np.linalg.norm(a - b))

    def _get_random_level(self):
        # UPDATE 2: Bitwise leading zero calculation
        
        # 1. Generate a random 32-bit unsigned integer
        r = random.getrandbits(32)
        
        # 2. Count leading zeros
        # r.bit_length() returns the number of bits needed to represent the integer.
        # Subtracting from 32 gives us the exact number of leading zeros.
        leading_zeros = 32 - r.bit_length()
        
        # 3. Scale by the bitwise multiplier and return as integer
        return int(leading_zeros * self.bitwise_level_mult)

    def _next_visit(self):
        self._visit_mark += 1
        if self._visit_mark % 1000000 == 0:
            self._visited.clear()

    def _is_visited(self, idx):
        return self._visited.get(idx, 0) == self._visit_mark

    def _set_visited(self, idx):
        self._visited[idx] = self._visit_mark

    def _greedy_search_layer(self, query_vec, entry_idx, level):
        current = entry_idx
        current_dist = self._distance(query_vec, self.nodes[current].vector)
        improved = True
        while improved:
            improved = False
            for nb in self.nodes[current].neighbors[level]:
                d = self._distance(query_vec, self.nodes[nb].vector)
                if d < current_dist:
                    current = nb
                    current_dist = d
                    improved = True
        return current

    def _search_layer(self, query_vec, entry_idx, ef, level):
        candidates = []
        result_heap = []

        self._next_visit()
        entry_dist = self._distance(query_vec, self.nodes[entry_idx].vector)
        heapq.heappush(candidates, (entry_dist, entry_idx))
        heapq.heappush(result_heap, (-entry_dist, entry_idx))
        self._set_visited(entry_idx)

        while candidates:
            top_dist, top_idx = candidates[0]
            worst_dist = -result_heap[0][0]
            if top_dist > worst_dist and len(result_heap) >= ef:
                break
            heapq.heappop(candidates)
            for nb in self.nodes[top_idx].neighbors[level]:
                if self._is_visited(nb):
                    continue
                self._set_visited(nb)
                d = self._distance(query_vec, self.nodes[nb].vector)
                if len(result_heap) < ef or d < -result_heap[0][0]:
                    heapq.heappush(candidates, (d, nb))
                    heapq.heappush(result_heap, (-d, nb))
                    if len(result_heap) > ef:
                        heapq.heappop(result_heap)

        res = [(-dist, idx) for (dist, idx) in result_heap]
        res.sort(key=lambda x: x[0])
        return res

    def _select_neighbors(self, query_vec, candidates, M):
        selected = []
        for dist_q, cand_idx in candidates:
            cand_vec = self.nodes[cand_idx].vector
            ok = True
            for sel_idx in selected:
                if self._distance(cand_vec, self.nodes[sel_idx].vector) < dist_q:
                    ok = False
                    break
            if ok:
                selected.append(cand_idx)
            if len(selected) >= M:
                break
        return selected

    def _select_neighbors_from_ids(self, query_vec, ids, M):
        cand_with_dist = [(self._distance(query_vec, self.nodes[idx].vector), idx) for idx in ids]
        cand_with_dist.sort(key=lambda x: x[0])
        return self._select_neighbors(query_vec, cand_with_dist, M)

    def add_item(self, vector, idx=None):
        if idx is None:
            idx = len(self.nodes)
        vector = np.asarray(vector, dtype=float)
        level = self._get_random_level()
        node = Node(idx, vector, level)
        if idx >= len(self.nodes):
            self.nodes.extend([None] * (idx - len(self.nodes) + 1))
        self.nodes[idx] = node

        if self.entry_point is None:
            self.entry_point = idx
            self.max_level = level
            return

        ep = self.entry_point
        for L in range(self.max_level, level, -1):
            ep = self._greedy_search_layer(vector, ep, L)

        for L in range(min(level, self.max_level), -1, -1):
            candidates = self._search_layer(vector, ep, self.ef_construction, L)
            selected = self._select_neighbors(vector, candidates, self.M)
            node.neighbors[L] = selected.copy()

            for nb in selected:
                nb_neighbors = self.nodes[nb].neighbors[L]
                nb_neighbors.append(idx)
                if len(nb_neighbors) > self.M:
                    pruned = self._select_neighbors_from_ids(self.nodes[nb].vector, nb_neighbors, self.M)
                    self.nodes[nb].neighbors[L] = pruned

            if candidates:
                ep = candidates[0][1]

        if level > self.max_level:
            self.entry_point = idx
            self.max_level = level

    def search_knn(self, query_vec, k=1, ef_search=None):
        if self.entry_point is None:
            return []
        if ef_search is None:
            ef_search = max(k, 50)

        ep = self.entry_point
        for L in range(self.max_level, 0, -1):
            ep = self._greedy_search_layer(query_vec, ep, L)
        results = self._search_layer(query_vec, ep, ef_search, 0)
        return results[:k]


In [4]:
from tqdm import tqdm
import numpy as np

def evaluate_hit_rate_topk(hnsw_index, index_labels, query_vectors, query_labels, k_max=50, ef_search=20):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)

    if total_indexed == 0 or len(query_labels) == 0:
        raise ValueError("Index or query set empty.")

    k_max = min(k_max, total_indexed)

    # Outer loop over k values with tqdm
    for k in tqdm(range(1, k_max + 1,10), desc="Evaluating top-K hit rates", position=0):
        hits = 0

        # Inner loop over queries with tqdm
        for qvec, qlabel in tqdm(zip(query_vectors, query_labels),
                                 total=len(query_labels),
                                 desc=f"K={k}",
                                 leave=False,
                                 position=1):
            res = hnsw_index.search_knn(qvec, k=k, ef_search=ef_search)
            if len(res) == 0:
                continue
            retrieved_idxs = [item[1] for item in res]
            retrieved_labels = index_labels[retrieved_idxs]
            if np.any(retrieved_labels == qlabel):
                hits += 1

        hit_rate = hits / len(query_labels)
        hit_rates.append(hit_rate)
        penetration = k / total_indexed
        x_labels.append(f"{k} ({penetration:.3f})")

        tqdm.write(f"K={k:3d}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates


In [5]:
def evaluate_hit_rate_efsearch(hnsw_index, index_labels, query_vectors, query_labels, ef_max=100):
    hit_rates = []
    x_labels = []
    total_indexed = len(index_labels)
    

    for ef in range(1, ef_max + 1):
        hits = 0
        for qvec, qlabel in zip(query_vectors, query_labels):
            res = hnsw_index.search_knn(qvec, k=1, ef_search=ef)
            if len(res) == 0:
                continue
            # res entries are (dist, idx)
            retrieved_idxs = [item[1] for item in res]
            retrieved_labels = index_labels[retrieved_idxs]
            if np.any(retrieved_labels == qlabel):
                hits += 1
        total = len(query_labels)
        hit_rate = hits / total if total > 0 else 0
        hit_rates.append(hit_rate)

        penetration = ef / total_indexed
        x_labels.append(f"{ef} ({penetration:.2f})")


        print(f"efSearch={ef}, Hit rate={hit_rate:.4f}")

    return x_labels, hit_rates

In [6]:
# =========================================================
#  Plot Hit Rate vs Penetration Rate
# =========================================================
def plot_hit_rate_plotly_topk(penetration_rates, hit_rates, efSearch,title_name="IITD_HNSW"):

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=penetration_rates,
        y=hit_rates,
        mode='lines+markers',
        name=f"efSearch={efSearch}",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    # numeric_vals = [
    #     float(re.search(r'\((.*?)\)', str(x)).group(1)) if '(' in str(x) else float(x)
    #     for x in penetration_rates
    # ]
    # max_tick = math.ceil(max(numeric_vals) * 100) / 100  # round up to next 0.01
    # # Clean tick marks at 0.01 intervals
    # tick_vals = np.round(np.arange(0.0, max_tick + 0.001, 0.01), 2).tolist()
    fig.update_xaxes(
        title='Penetration Rate (k / total indexed)',
        # tickvals=tick_vals,
        # tickformat=".2f"
    )
    fig.update_yaxes(title='Hit Rate')

    fig.update_layout(
        title=f"{title_name}",
        template='plotly_white',
        width=800,
        height=500,
        hovermode="x unified"
    )
    fig.show()


In [8]:
def plot_hit_rate_plotly_efsearch(x_labels, hit_rates,title="IITDV1 HNSW efSearch"):
    fig = go.Figure()

    # Main line
    fig.add_trace(go.Scatter(
        x=x_labels,
        y=hit_rates,
        mode='lines+markers',
        name="Hit Rate",
        line=dict(width=2),
        marker=dict(size=6)
    ))

    fig.update_layout(
        title=title,
        xaxis_title="efSearch (Penetration)",
        yaxis_title="Hit Rate",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

In [9]:
import time
import numpy as np
from typing import Tuple, List, Dict

def evaluate_top1_timing(hnsw_index,
                         index_labels: np.ndarray,
                         query_vectors: np.ndarray,
                         query_labels: np.ndarray,
                         ef_search: int = 20) -> Dict[str, object]:
    """
    For each query vector:
      - perform a top-1 search (k=1)
      - measure the time taken for the search call
      - check if the retrieved top-1 label matches the query label

    Returns a dict containing:
      - 'hit_rate': fraction of queries whose top-1 label matched
      - 'avg_time_s': average search time per query in seconds
      - 'times_s': list of per-query times (seconds) in the same order as query_vectors
      - 'total_queries': number of queries
      - 'successful_searches': number of searches that returned at least one neighbor
    """
    if len(index_labels) == 0 or len(query_labels) == 0:
        raise ValueError("Index or query set empty.")

    if len(query_vectors) != len(query_labels):
        raise ValueError("query_vectors and query_labels must have the same length.")

    times: List[float] = []
    hits = 0
    successful_searches = 0
    total_queries = len(query_labels)

    for qvec, qlabel in zip(query_vectors, query_labels):
        t0 = time.perf_counter()
        res = hnsw_index.search_knn(qvec, k=1, ef_search=ef_search)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        times.append(elapsed)

        # If the index returned at least one neighbor, check label
        if len(res) > 0:
            successful_searches += 1
            # assume res is iterable of (distance, index) or similar
            # take the index of the first (top-1) result
            top_idx = res[0][1]
            if index_labels[top_idx] == qlabel:
                hits += 1

    hit_rate = hits / total_queries
    avg_time_s = float(sum(times) / total_queries) if total_queries > 0 else 0.0

    return {
        "hit_rate": hit_rate,
        "avg_time_s": avg_time_s,
        "times_s": times,
        "total_queries": total_queries,
        "successful_searches": successful_searches
    }


# IITDV1

In [11]:
# ---------------------
# Load IITD_UPD Templates (70:30 Split) - as provided by you
# ---------------------
def load_iitd_upd_templates(root_folder, seed=42, split_ratio=0.7):
    """
    For each subject folder (like 1_L, 2_L...), randomly split its .npy files
      - 70% for indexing (gallery/enrollment)
      - 30% for query (probe)
    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    random.seed(seed)
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    subject_folders = sorted(
        [d for d in os.listdir(root_folder) if os.path.isdir(os.path.join(root_folder, d))]
    )

    for subject in subject_folders:
        subject_path = os.path.join(root_folder, subject)
        files = sorted([f for f in os.listdir(subject_path) if f.endswith('.npy')])

        n = len(files)
        if n == 0:
            print(f"⚠️ Skipping {subject}, no .npy files found.")
            continue

        random.shuffle(files)
        split_point = max(1, int(n * split_ratio))  # At least 1 for indexing
        index_files = files[:split_point]
        query_files = files[split_point:] if split_point < n else []

        # Load index vectors
        for f in index_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            index_vectors.append(vec)
            index_labels.append(subject)

        # Load query vectors
        for f in query_files:
            vec = np.load(os.path.join(subject_path, f)).astype('float32').flatten()
            query_vectors.append(vec)
            query_labels.append(subject)

        print(f"{subject}: {len(index_files)} index, {len(query_files)} query")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check `root_folder` and structure.")

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors) if len(query_vectors) > 0 else np.zeros((0, index_vectors[0].shape[0]), dtype='float32'),
        np.array(query_labels)
    )


In [37]:
root = "/home/nishkal/sg/iris_indexing/IITD_UPD"
idx_vecs, idx_labels, qry_vecs, qry_labels = load_iitd_upd_templates(root, seed=42)
print("Index:", idx_vecs.shape, idx_labels.shape)
print("Query:", qry_vecs.shape, qry_labels.shape)

100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [15]:

data_folder = "/home/nishkal/sg/iris_indexing/IITD_UPD"
X_index, y_index, X_query, y_query = load_iitd_upd_templates(data_folder, seed=42)

hnsw = HNSW(M=16, ef_construction=200, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

x_labels, hit_rates = evaluate_hit_rate_efsearch(hnsw, y_index, X_query, y_query)

plot_hit_rate_plotly_efsearch(x_labels, hit_rates,"IITDV1 HNSW efSearch")

100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [12]:
data_folder = "/home/nishkal/sg/iris_indexing/IITD_UPD"
X_index, y_index, X_query, y_query = load_iitd_upd_templates(data_folder, seed=42)

hnsw = HNSW(M=16, ef_construction=200, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

100_L: 3 index, 2 query
100_R: 3 index, 2 query
101_L: 3 index, 2 query
101_R: 3 index, 2 query
102_L: 3 index, 2 query
102_R: 3 index, 2 query
103_L: 3 index, 2 query
103_R: 3 index, 2 query
104_L: 3 index, 2 query
104_R: 3 index, 2 query
105_L: 3 index, 2 query
105_R: 3 index, 2 query
106_L: 3 index, 2 query
106_R: 3 index, 2 query
107_L: 3 index, 2 query
107_R: 3 index, 2 query
108_L: 3 index, 2 query
108_R: 3 index, 2 query
109_L: 3 index, 2 query
109_R: 3 index, 2 query
10_L: 7 index, 3 query
110_L: 3 index, 2 query
110_R: 3 index, 2 query
111_L: 3 index, 2 query
111_R: 3 index, 2 query
112_L: 3 index, 2 query
112_R: 3 index, 2 query
113_L: 3 index, 2 query
113_R: 3 index, 2 query
114_L: 3 index, 2 query
114_R: 3 index, 2 query
115_L: 3 index, 2 query
115_R: 3 index, 2 query
116_L: 3 index, 2 query
116_R: 3 index, 2 query
117_L: 3 index, 2 query
117_R: 3 index, 2 query
118_L: 3 index, 2 query
118_R: 3 index, 2 query
119_L: 3 index, 2 query
119_R: 3 index, 2 query
11_L: 7 index, 3 

In [26]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=23)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.9048697621744054
Average search time (ms): 0.5050659209219812


In [13]:


x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 50,ef_search=23)



Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:21,  2.26it/s]

K=  1, Hit rate=0.9049


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:00<00:21,  2.27it/s]

K=  2, Hit rate=0.9434


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:01<00:20,  2.28it/s]

...

K= 48, Hit rate=0.9796


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:21<00:00,  2.28it/s]

K= 49, Hit rate=0.9796


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:22<00:00,  2.27it/s]

K= 50, Hit rate=0.9796


In [14]:
np.savez('IITDv1_hnsw.npz', x=x_labels, y=hit_rates)

In [30]:
plot_hit_rate_plotly_topk(x_labels, hit_rates,23,"IITDV1 HNSW")

# Iris_Lamp

In [10]:
data = np.load("/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates/001_L_01_template.npz")

print("Keys:", data.files)

Keys: ['iris_code', 'mask_code']


In [11]:

def load_IrisLamp_templates(root_folder, seed=42, pick='random'):
    """
    Process a flat folder of template files named like: 001_L_01_template.npz
    Groups by subject-eye (e.g. '001_L'), picks exactly one file per group for indexing,
    and the rest for querying.

    Args:
        root_folder (str): path to folder containing .npz / .npy template files.
        seed (int): deterministic seed for random picking.
        pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.

    Returns:
        index_vectors: np.ndarray (M, D) float32
        index_labels:  np.ndarray (M,) strings like '001_L'
        query_vectors: np.ndarray (Q, D) float32 (empty if none)
        query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # regex to extract subject and eye from filename; flexible to variations
    # examples matched: 001_L_01_template.npz, 23_R_3_template.npz, 0001_L_12_template.npz
    pattern = re.compile(r'^(\d+)_([LRlr])_.*template', re.IGNORECASE)

    # collect files
    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}  # key -> list of filenames
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            # skip files that don't match pattern (or optionally group them by filename prefix)
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)  # pad subject id to 3 digits for consistent labels
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    # helper to convert a single file into a 1D float32 normalized vector
    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                # try broadcasting or raise
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            # mask==1 => occluded/unreliable; set corresponding bits to 0
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize if possible
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors = []
    index_labels = []
    query_vectors = []
    query_labels = []

    # iterate groups
    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        # pick first as index, rest are query
        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


    


In [12]:
root = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
idx_vecs, idx_labels, qry_vecs, qry_labels = load_IrisLamp_templates(root, seed=123, pick='random')
print("Index:", idx_vecs.shape, idx_labels.shape)
print("Query:", qry_vecs.shape, qry_labels.shape)

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [45]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisLamp_templates(data_folder, seed=42)

def create_hnsw_index(vectors, M=12, ef_construction=30, ef_search=15):
    """
    Create an HNSW index with FAISS.
    """
    d = vectors.shape[1]
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = ef_construction
    index.add(vectors)
    index.hnsw.efSearch = ef_search
    return index


hnsw = HNSW(M=16, ef_construction=100, random_seed=42)
for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

x_labels, hit_rates = evaluate_hit_rate_efsearch(hnsw, y_index, X_query, y_query)

plot_hit_rate_plotly_efsearch(x_labels, hit_rates,"IRISLAMP1 HNSW efSearch")

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [50]:
x_labels, hit_rates = evaluate_hit_rate_efsearch(hnsw, y_index, X_query, y_query,200)

# print(hit_rates)

efSearch=200, Hit rate=0.8643


In [13]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisLamp_templates(data_folder, seed=42)

hnsw2 = HNSW2(M=16, ef_construction=200, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw2.add_item(vec, idx=i)

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [16]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Lamp_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisLamp_templates(data_folder, seed=42)

hnsw = HNSW(M=16, ef_construction=200, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

# x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 100,ef_search=150)

# plot_hit_rate_plotly_topk(x_labels, hit_rates,150,"CASIAV4-IRIS_LAMP HNSW")

001_L: index=1, query=19
001_R: index=1, query=19
002_L: index=1, query=19
002_R: index=1, query=19
003_L: index=1, query=19
003_R: index=1, query=19
004_L: index=1, query=19
004_R: index=1, query=19
005_L: index=1, query=19
005_R: index=1, query=19
006_L: index=1, query=19
006_R: index=1, query=19
007_L: index=1, query=19
007_R: index=1, query=19
008_L: index=1, query=19
008_R: index=1, query=19
009_L: index=1, query=19
009_R: index=1, query=19
010_L: index=1, query=19
010_R: index=1, query=19
011_L: index=1, query=19
011_R: index=1, query=19
012_L: index=1, query=19
012_R: index=1, query=19
013_L: index=1, query=19
013_R: index=1, query=19
014_L: index=1, query=19
014_R: index=1, query=19
015_L: index=1, query=19
015_R: index=1, query=19
016_L: index=1, query=19
016_R: index=1, query=19
017_L: index=1, query=19
017_R: index=1, query=19
018_L: index=1, query=19
018_R: index=1, query=19
019_L: index=1, query=19
019_R: index=1, query=19
020_L: index=1, query=19
020_R: index=1, query=19


In [14]:
results = evaluate_top1_timing(hnsw2, y_index, X_query, y_query,ef_search=150)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.8614963860128931
Average search time (ms): 9.739310549449161


In [23]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=150)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.8614963860128931
Average search time (ms): 10.25060343621087


In [18]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 100,ef_search=150)

np.savez('IrisLamp_hnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   5%|▌         | 1/20 [02:30<47:30, 150.02s/it]

K=  1, Hit rate=0.8615


Evaluating top-K hit rates:  10%|█         | 2/20 [05:04<45:51, 152.85s/it]

K=  6, Hit rate=0.8879


Evaluating top-K hit rates:  15%|█▌        | 3/20 [07:33<42:44, 150.87s/it]

...

K= 86, Hit rate=0.9238


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [51:27<02:46, 166.96s/it]

K= 91, Hit rate=0.9249


Evaluating top-K hit rates: 100%|██████████| 20/20 [54:09<00:00, 162.49s/it]

K= 96, Hit rate=0.9260


In [10]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 100,ef_search=150)

plot_hit_rate_plotly_topk(x_labels, hit_rates,150,"IRIS_LAMP HNSW")

Evaluating top-K hit rates:   5%|▌         | 1/20 [02:30<47:44, 150.76s/it]

K=  1, Hit rate=0.8615


Evaluating top-K hit rates:  10%|█         | 2/20 [04:58<44:44, 149.17s/it]

K=  6, Hit rate=0.8879


Evaluating top-K hit rates:  15%|█▌        | 3/20 [07:27<42:15, 149.17s/it]

...

K= 86, Hit rate=0.9238


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [48:54<02:38, 158.20s/it]

K= 91, Hit rate=0.9249


Evaluating top-K hit rates: 100%|██████████| 20/20 [51:33<00:00, 154.66s/it]


K= 96, Hit rate=0.9260


In [14]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 100,ef_search=150)

K= 44, Hit rate=0.9140
K= 54, Hit rate=0.9169
K= 64, Hit rate=0.9191
K= 74, Hit rate=0.9211
K= 84, Hit rate=0.9235
K= 94, Hit rate=0.9255


In [5]:
import plotly.graph_objects as go

# Data
K = list(range(1, 34)) + [44, 54, 64, 74, 84, 94]
hit_rate = [
    0.8615, 0.8730, 0.8782, 0.8821, 0.8857, 0.8879, 0.8896, 0.8913, 0.8933, 0.8946,
    0.8958, 0.8973, 0.8983, 0.8993, 0.9005, 0.9012, 0.9017, 0.9024, 0.9028, 0.9036,
    0.9041, 0.9047, 0.9053, 0.9055, 0.9061, 0.9067, 0.9073, 0.9078, 0.9084, 0.9091,
    0.9095, 0.9098, 0.9104,
    0.9140, 0.9169, 0.9191, 0.9211, 0.9235, 0.9255
]

# Define tick positions and labels
tickvals = [k for k in range(1, 35) if (k - 1) % 5 == 0] + [44, 54, 64, 74, 84, 94]
ticktext = [
    f"{k}" if k < 44 else f"{k} ({k/819:.3f})"
    for k in tickvals
]

# Create figure
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=K,
    y=hit_rate,
    mode='lines+markers',
    marker=dict(size=6),
    line=dict(width=2),
    name='Hit Rate'
))

# Layout customization
fig.update_layout(
    title='Iris_Lamp HNSW',
    xaxis_title='K (Penetration Rate)',
    yaxis_title='Hit Rate',
    xaxis=dict(
        tickmode='array',
        tickvals=tickvals,
        ticktext=ticktext,
        showgrid=True,
        tickangle=45
    ),
    yaxis=dict(range=[0.86, 0.93], showgrid=True),
    template='plotly_white',
    font=dict(size=14)
)

fig.show()


# CASIA V1

In [7]:
def load_CASIAV1_templates(data_folder):
    """
    Load CASIA V1 templates according to the structure:
        <subjectID>_<session>_<sample>.jpg.mat

    Logic:
        - Files ending with '_3.jpg.mat' or '_4.jpg.mat' → Query templates
        - Files ending with '_1.jpg.mat' or '_2.jpg.mat' → Index (enrollment) templates

    Returns:
        index_vectors, index_labels, query_vectors, query_labels
    """
    files = sorted([f for f in os.listdir(data_folder) if f.endswith('.jpg.mat')])
    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for file in files:
        path = os.path.join(data_folder, file)
        data = loadmat(path)
        vec = data['template'].flatten().astype('float32')

        # Extract identifiers
        subject_id, session, sample = file.replace('.jpg.mat', '').split('_')
        label = f"{subject_id}_{session}"  # subject-session pair label

        if sample in ('3', '4'):  # Probe (query)
            query_vectors.append(vec)
            query_labels.append(label)
        else:  # Enrollment (index)
            index_vectors.append(vec)
            index_labels.append(label)

    return (
        np.vstack(index_vectors),
        np.array(index_labels),
        np.vstack(query_vectors),
        np.array(query_labels)
    )


In [5]:
root = "/home/nishkal/sg/iris_indexing/Updated_CASIAV1"
idx_vecs, idx_labels, qry_vecs, qry_labels = load_CASIAV1_templates(root)
print("Index:", idx_vecs.shape, idx_labels.shape)
print("Query:", qry_vecs.shape, qry_labels.shape)

Index: (432, 9600) (432,)
Query: (324, 9600) (324,)


In [8]:
data_folder = "/home/nishkal/sg/iris_indexing/Updated_CASIAV1"
X_index, y_index, X_query, y_query = load_CASIAV1_templates(data_folder)

hnsw = HNSW(M=8, ef_construction=100, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

In [35]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=20)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.5648148148148148
Average search time (ms): 1.9494328636354135


In [9]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 50,ef_search=20)

Evaluating top-K hit rates:   2%|▏         | 1/50 [00:00<00:31,  1.53it/s]

K=  1, Hit rate=0.5648


Evaluating top-K hit rates:   4%|▍         | 2/50 [00:01<00:31,  1.52it/s]

K=  2, Hit rate=0.7685


Evaluating top-K hit rates:   6%|▌         | 3/50 [00:01<00:31,  1.51it/s]

...

K= 48, Hit rate=0.9228


Evaluating top-K hit rates:  98%|█████████▊| 49/50 [00:30<00:00,  1.66it/s]

K= 49, Hit rate=0.9228


Evaluating top-K hit rates: 100%|██████████| 50/50 [00:31<00:00,  1.59it/s]

K= 50, Hit rate=0.9228


In [10]:
np.savez('CASIAv1_hnsw.npz', x=x_labels, y=hit_rates)

In [34]:
plot_hit_rate_plotly_topk(x_labels, hit_rates,20,"CASIAV1 HNSW")

# Iris Thousand

In [19]:
import os
import re
import random
import numpy as np

def load_IrisThousand_templates(root_folder, seed=42, pick='random', expected_per_group=10):
    """
    Loader for 'Iris Thousand' style filenames like:
      000_L_00_template.npz ... 000_L_09_template.npz
      000_R_00_template.npz ... 000_R_09_template.npz
      ...
      999_R_09_template.npz

    Behavior:
      - Groups files by subject (0..999) and eye (L/R).
      - Picks exactly one file per group as the index vector (strategy: 'random' or 'first').
      - All remaining files in that group become queries for that label.

    Args:
      root_folder (str): path to folder containing .npz / .npy template files.
      seed (int): random seed for deterministic shuffling.
      pick (str): 'random' (default) or 'first' - selection strategy for the 1 index file.
      expected_per_group (int): expected images per subject-eye (default 10). Used to warn if mismatch.

    Returns:
      index_vectors: np.ndarray (M, D) float32
      index_labels:  np.ndarray (M,) strings like '000_L'
      query_vectors: np.ndarray (Q, D) float32 (empty if none)
      query_labels:  np.ndarray (Q,) strings
    """
    random.seed(seed)
    np.random.seed(seed)

    # Pattern extracts subject (1-3 digits), eye (L/R) and image number (00..09 or similar)
    pattern = re.compile(r'^(\d{1,3})_([LRlr])_0*(\d+)_template', re.IGNORECASE)

    files = [f for f in os.listdir(root_folder) if f.lower().endswith(('.npz', '.npy'))]
    if not files:
        raise ValueError(f"No .npz/.npy files found in {root_folder}")

    groups = {}
    for f in sorted(files):
        m = pattern.match(f)
        if not m:
            print(f"⚠️ Skipping file with unexpected name format: {f}")
            continue
        subj = m.group(1).zfill(3)   # ensure three-digit subject id like '000'
        eye = m.group(2).upper()
        key = f"{subj}_{eye}"
        groups.setdefault(key, []).append(f)

    def file_to_vector(path):
        full = os.path.join(root_folder, path)
        if full.lower().endswith('.npz'):
            data = np.load(full, allow_pickle=True)
            # keep the iris_code / mask_code logic you already used
            if 'iris_code' not in data.files or 'mask_code' not in data.files:
                raise ValueError(f"{path} missing 'iris_code' or 'mask_code' keys.")
            iris_code = np.array(data['iris_code']).reshape(-1)
            mask_code = np.array(data['mask_code']).reshape(-1)
            if iris_code.shape != mask_code.shape:
                if iris_code.size == mask_code.size:
                    pass
                else:
                    raise ValueError(f"Shape mismatch in {path}: iris_code {iris_code.shape}, mask_code {mask_code.shape}")
            vec = np.where(mask_code == 1, 0, iris_code).astype(np.float32)
        elif full.lower().endswith('.npy'):
            vec = np.load(full).reshape(-1).astype(np.float32)
        else:
            raise ValueError(f"Unsupported file: {path}")

        # L2 normalize
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        return vec

    index_vectors, index_labels = [], []
    query_vectors, query_labels = [], []

    for key in sorted(groups.keys()):
        group_files = groups[key][:]
        if len(group_files) == 0:
            continue

        if len(group_files) != expected_per_group:
            print(f"⚠️ Warning: group {key} has {len(group_files)} files (expected {expected_per_group}).")

        if pick == 'random':
            random.shuffle(group_files)
        elif pick == 'first':
            group_files = sorted(group_files)
        else:
            raise ValueError("pick must be 'random' or 'first'")

        idx_file = group_files[0]
        try:
            idx_vec = file_to_vector(idx_file)
        except Exception as e:
            print(f"⚠️ Failed to load index file {idx_file} for {key}: {e}. Skipping this group.")
            continue

        index_vectors.append(idx_vec)
        index_labels.append(key)

        for qf in group_files[1:]:
            try:
                qvec = file_to_vector(qf)
            except Exception as e:
                print(f"⚠️ Failed to load query file {qf} for {key}: {e}. Skipping this file.")
                continue
            query_vectors.append(qvec)
            query_labels.append(key)

        print(f"{key}: index=1, query={max(0, len(group_files)-1)}")

    if len(index_vectors) == 0:
        raise ValueError("No index vectors found. Check filename patterns and folder contents.")

    index_vectors = np.vstack(index_vectors).astype(np.float32)
    index_labels = np.array(index_labels)

    if len(query_vectors) > 0:
        query_vectors = np.vstack(query_vectors).astype(np.float32)
        query_labels = np.array(query_labels)
    else:
        D = index_vectors.shape[1]
        query_vectors = np.zeros((0, D), dtype=np.float32)
        query_labels = np.array([])

    return index_vectors, index_labels, query_vectors, query_labels


In [21]:
data_folder = "/home/nishkal/sg/iris_indexing/CASIA-Iris-Thousand_/outputs_npz/templates"
X_index, y_index, X_query, y_query = load_IrisThousand_templates(data_folder, seed=42)

hnsw = HNSW(M=16, ef_construction=200, random_seed=42)

for i, vec in enumerate(X_index):
        hnsw.add_item(vec, idx=i)

⚠️ Warning: group 000_L has 9 files (expected 10).
000_L: index=1, query=8
000_R: index=1, query=9
001_L: index=1, query=9
⚠️ Warning: group 001_R has 9 files (expected 10).
001_R: index=1, query=8
002_L: index=1, query=9
⚠️ Warning: group 002_R has 8 files (expected 10).
002_R: index=1, query=7
003_L: index=1, query=9
003_R: index=1, query=9
004_L: index=1, query=9
004_R: index=1, query=9
005_L: index=1, query=9
005_R: index=1, query=9
⚠️ Warning: group 006_L has 7 files (expected 10).
006_L: index=1, query=6
006_R: index=1, query=9
007_L: index=1, query=9
⚠️ Warning: group 007_R has 7 files (expected 10).
007_R: index=1, query=6
008_L: index=1, query=9
008_R: index=1, query=9
009_L: index=1, query=9
009_R: index=1, query=9
⚠️ Warning: group 010_L has 8 files (expected 10).
010_L: index=1, query=7
⚠️ Warning: group 010_R has 7 files (expected 10).
010_R: index=1, query=6
011_L: index=1, query=9
011_R: index=1, query=9
⚠️ Warning: group 012_L has 9 files (expected 10).
012_L: index=1, 

In [24]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 200,ef_search=200)

np.savez('IrisThousand_hnsw.npz', x=x_labels, y=hit_rates)

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:12<1:38:53, 312.30s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:27<1:34:09, 313.86s/it]

K= 11, Hit rate=0.6729


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:38<1:28:37, 312.81s/it]

...

K=171, Hit rate=0.7480


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:39:20<05:13, 313.38s/it]

K=181, Hit rate=0.7495


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:44:35<00:00, 313.80s/it]

K=191, Hit rate=0.7524


In [14]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 100,ef_search=200)

plot_hit_rate_plotly_topk(x_labels, hit_rates,200,"IRIS_Thousand HNSW")

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:07<1:37:29, 307.89s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:18<1:32:53, 309.63s/it]

K=  6, Hit rate=0.6588


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:31<1:28:10, 311.21s/it]

...

K= 86, Hit rate=0.7257


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:38:20<05:10, 310.74s/it]

K= 91, Hit rate=0.7275


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:43:29<00:00, 310.47s/it]

K= 96, Hit rate=0.7297


In [17]:
x_labels, hit_rates = evaluate_hit_rate_topk(hnsw, y_index, X_query, y_query,k_max = 200,ef_search=200)

plot_hit_rate_plotly_topk(x_labels, hit_rates,200,"IRIS_Thousand HNSW")

Evaluating top-K hit rates:   5%|▌         | 1/20 [05:06<1:36:56, 306.12s/it]

K=  1, Hit rate=0.6159


Evaluating top-K hit rates:  10%|█         | 2/20 [10:13<1:32:00, 306.68s/it]

K= 11, Hit rate=0.6729


Evaluating top-K hit rates:  15%|█▌        | 3/20 [15:19<1:26:49, 306.44s/it]

...

K=171, Hit rate=0.7480


Evaluating top-K hit rates:  95%|█████████▌| 19/20 [1:36:52<05:05, 305.39s/it]

K=181, Hit rate=0.7495


Evaluating top-K hit rates: 100%|██████████| 20/20 [1:41:58<00:00, 305.92s/it]

K=191, Hit rate=0.7524


In [ ]:
results = evaluate_top1_timing(hnsw, y_index, X_query, y_query,ef_search=200)
print("Hit rate:", results["hit_rate"])
print("Average search time (ms):", results["avg_time_s"] * 1000)

Hit rate: 0.6159398313823012
Average search time (ms): 18.19881511702988
